In [1]:
import pandas as pd
df = pd.read_csv(
    "../data/processed/matches_bronze.csv"
)

df.shape

(49505, 9)

In [2]:
df["date"] = pd.to_datetime(df["date"])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49505 entries, 0 to 49504
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        49505 non-null  datetime64[ns]
 1   home_team   49505 non-null  object        
 2   away_team   49505 non-null  object        
 3   home_score  49501 non-null  float64       
 4   away_score  49501 non-null  float64       
 5   tournament  49505 non-null  object        
 6   city        49505 non-null  object        
 7   country     49505 non-null  object        
 8   neutral     49505 non-null  bool          
dtypes: bool(1), datetime64[ns](1), float64(2), object(5)
memory usage: 3.1+ MB


In [3]:

    silver_matches = df.dropna(
    subset=["home_score", "away_score"]
)

In [4]:
silver_matches.duplicated().sum()

np.int64(0)

In [5]:
silver_matches = silver_matches.reset_index(drop=True)

silver_matches["match_id"] = (
    silver_matches.index + 1
)
silver_matches.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,match_id
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,2
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,3
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,4
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,5


In [6]:
silver_matches = silver_matches[
    [
        "match_id",
        "date",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "tournament",
        "city",
        "country",
        "neutral"
    ]
]
silver_matches.columns

Index(['match_id', 'date', 'home_team', 'away_team', 'home_score',
       'away_score', 'tournament', 'city', 'country', 'neutral'],
      dtype='object')

CREACION DE IDENTIFICADOR UNICO PARA AUTOMATIZACION DE PARTIDOS


In [7]:
silver_matches["match_key"] = (
    silver_matches["date"].dt.strftime("%Y-%m-%d")
    + "|"
    + silver_matches["home_team"]
    + "|"
    + silver_matches["away_team"]
    + "|"
    + silver_matches["tournament"]
)

In [8]:
silver_matches[
    ["date", "home_team", "away_team", "tournament", "match_key"]
].head()

,date,home_team,away_team,tournament,match_key
0,1872-11-30,Scotland,England,Friendly,1872-11-30|Scotland|England|Friendly
1,1873-03-08,England,Scotland,Friendly,1873-03-08|England|Scotland|Friendly
2,1874-03-07,Scotland,England,Friendly,1874-03-07|Scotland|England|Friendly
3,1875-03-06,England,Scotland,Friendly,1875-03-06|England|Scotland|Friendly
4,1876-03-04,Scotland,England,Friendly,1876-03-04|Scotland|England|Friendly


In [9]:
silver_matches["match_key"] = (
    silver_matches["date"].dt.strftime("%Y-%m-%d")
    + "|"
    + silver_matches["home_team"]
    + "|"
    + silver_matches["away_team"]
    + "|"
    + silver_matches["tournament"]
    + "|"
    + silver_matches["home_score"].astype(str)
    + "|"
    + silver_matches["away_score"].astype(str)
    + "|"
    + silver_matches["city"]
)

In [10]:
duplicate_match_keys = silver_matches[
    silver_matches["match_key"].duplicated(keep=False)
].sort_values("match_key")

print("Match keys duplicadas:", duplicate_match_keys["match_key"].nunique())
print("Registros involucrados:", len(duplicate_match_keys))

Match keys duplicadas: 0
Registros involucrados: 0


In [11]:
mexico_silver = silver_matches[
    (silver_matches["home_team"] == "Mexico") |
    (silver_matches["away_team"] == "Mexico")
].copy()

print("Partidos de México:", len(mexico_silver))
print("Match keys:", mexico_silver["match_key"].notna().sum())

Partidos de México: 1008
Match keys: 1008


In [12]:
gold_match_ids = set(mexico_silver["match_id"])

print("Match IDs de México:", len(gold_match_ids))
print("Match IDs únicos:", mexico_silver["match_id"].nunique())

Match IDs de México: 1008
Match IDs únicos: 1008


In [13]:
mexico_silver.to_csv(
    "../data/processed/mexico_silver.csv",
    index=False
)

print("Partidos de México guardados:", len(mexico_silver))

Partidos de México guardados: 1008


In [14]:
print("Registros:", len(mexico_silver))
print("Match keys únicas:", mexico_silver["match_key"].nunique())
print("Match keys nulas:", mexico_silver["match_key"].isna().sum())

Registros: 1008
Match keys únicas: 1008
Match keys nulas: 0


In [15]:
silver_matches = silver_matches[
    [
        "match_id",
        "match_key",
        "date",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "tournament",
        "city",
        "country",
        "neutral"
    ]
]

silver_matches.columns

Index(['match_id', 'match_key', 'date', 'home_team', 'away_team', 'home_score',
       'away_score', 'tournament', 'city', 'country', 'neutral'],
      dtype='object')

In [16]:
print(silver_matches.columns.tolist())

['match_id', 'match_key', 'date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']


In [17]:
silver_matches.to_csv(
    "../data/processed/matches_silver.csv",
    index=False
)